# STIR-Net V1 — first real backward gate

This notebook starts from the artifacts produced by `02_prepare_trackastra_first_overfit.ipynb` and uses the **current hardened STIR-Net source code**.

It intentionally contains **no runtime monkey patches**.

The goal is limited to:

1. rebuild the same all-cell BlastoSPIM sample used by the successful forward acceptance test;
2. reproduce one finite forward + full loss;
3. run **one AMP backward pass**;
4. verify gradients are finite and non-zero;
5. run one optimizer step and confirm a parameter changed;
6. record peak CUDA memory.

This is **not** the 100-step overfit experiment yet.


In [ ]:
from pathlib import Path
import gc
import json
import pickle
import time

import numpy as np
import torch

from learned.stirnet import RefinementCriterion, StirNet
from learned.stirnet.data.targets import build_gt_targets, extract_instance_metadata
from learned.stirnet.training.trainer import model_forward_from_batch, move_to_device
from learned.stirnet.debugging.acceptance.first_overfit import (
    _build_temporal_inputs,
    _reduced_config,
    _repo_root,
    _roi_with_all_cells,
)

REPO_ROOT = _repo_root(Path.cwd())

DATA_DIR = (
    REPO_ROOT
    / "data"
    / "learned"
    / "stirnet"
    / "first_overfit"
    / "BlastoSPIM1_F22_030_034"
)

TRACKASTRA_DIR = DATA_DIR / "trackastra"
SOURCE_DIR = DATA_DIR / "stirnet_source"

TARGET_LOCAL_T = 2
ROI_MARGIN_UM = 12.0

if not torch.cuda.is_available():
    raise RuntimeError("This backward gate requires CUDA.")

device = torch.device("cuda")

print("Repository :", REPO_ROOT)
print("Data       :", DATA_DIR)
print("GPU        :", torch.cuda.get_device_name(0))
print("CUDA       :", torch.version.cuda)
print("PyTorch    :", torch.__version__)


## 1. Preflight

In [ ]:
required_files = [
    DATA_DIR / "raw_movie.npy",
    DATA_DIR / "instance_movie.npy",
    DATA_DIR / "markers_movie.npy",
    DATA_DIR / "gt_movie.npy",
    DATA_DIR / "metadata.json",
    TRACKASTRA_DIR / "track_graph.pkl",
    SOURCE_DIR / "raw_norm_target.npy",
    SOURCE_DIR / "foreground_target.npy",
    SOURCE_DIR / "edt_target.npy",
    SOURCE_DIR / "boundary_target.npy",
    SOURCE_DIR / "marker_heatmap_target.npy",
    SOURCE_DIR / "dref_um.npy",
]

missing = [path for path in required_files if not path.exists()]

if missing:
    raise FileNotFoundError(
        "The first-overfit preparation cache is incomplete. "
        "Run notebooks/stirnet/02_prepare_trackastra_first_overfit.ipynb first.\n\n"
        + "\n".join(str(path) for path in missing)
    )

print("Preparation cache is complete.")


## 2. Load saved artifacts

In [ ]:
raw_movie = np.load(DATA_DIR / "raw_movie.npy", mmap_mode="r")
instance_movie = np.load(DATA_DIR / "instance_movie.npy", mmap_mode="r")
markers_movie = np.load(DATA_DIR / "markers_movie.npy", mmap_mode="r")
gt_movie = np.load(DATA_DIR / "gt_movie.npy", mmap_mode="r")

with (DATA_DIR / "metadata.json").open("r", encoding="utf-8") as handle:
    metadata = json.load(handle)

with (TRACKASTRA_DIR / "track_graph.pkl").open("rb") as handle:
    track_graph = pickle.load(handle)

spacing = np.asarray(metadata["spacing_zyx_um"], dtype=np.float32)
dref_um = float(np.load(SOURCE_DIR / "dref_um.npy"))

print("Movie shape :", raw_movie.shape)
print("Spacing um  :", tuple(float(v) for v in spacing))
print("dref um     :", dref_um)
print("Graph nodes :", track_graph.number_of_nodes())
print("Graph edges :", track_graph.number_of_edges())


## 3. Build the all-cell ROI

Only empty acquisition margins are removed. The assertions below verify that **all current and GT cells remain in the logical sample**.


In [ ]:
roi, roi_low, roi_high = _roi_with_all_cells(
    instance_movie,
    gt_movie,
    spacing,
    margin_um=ROI_MARGIN_UM,
)

roi_shape = tuple((roi_high - roi_low).tolist())

current_target = np.asarray(
    instance_movie[TARGET_LOCAL_T][roi]
).astype(np.int32, copy=True)

gt_target = np.asarray(
    gt_movie[TARGET_LOCAL_T][roi]
).astype(np.int32, copy=True)

current_count_full = int(
    np.count_nonzero(np.unique(instance_movie[TARGET_LOCAL_T]) > 0)
)
current_count_roi = int(
    np.count_nonzero(np.unique(current_target) > 0)
)

gt_count_full = int(
    np.count_nonzero(np.unique(gt_movie[TARGET_LOCAL_T]) > 0)
)
gt_count_roi = int(
    np.count_nonzero(np.unique(gt_target) > 0)
)

assert current_count_roi == current_count_full
assert gt_count_roi == gt_count_full

print("ROI low/high :", tuple(roi_low), tuple(roi_high))
print("ROI shape    :", roi_shape)
print("Current cells:", current_count_roi)
print("GT cells     :", gt_count_roi)


## 4. Build the five spatial input channels

In [ ]:
spatial_inputs = np.stack(
    [
        np.asarray(np.load(SOURCE_DIR / "raw_norm_target.npy", mmap_mode="r")[roi]),
        np.asarray(np.load(SOURCE_DIR / "foreground_target.npy", mmap_mode="r")[roi]),
        np.asarray(np.load(SOURCE_DIR / "edt_target.npy", mmap_mode="r")[roi]),
        np.asarray(np.load(SOURCE_DIR / "boundary_target.npy", mmap_mode="r")[roi]),
        np.asarray(np.load(SOURCE_DIR / "marker_heatmap_target.npy", mmap_mode="r")[roi]),
    ],
    axis=0,
).astype(np.float32, copy=False)

print("Spatial inputs:", spatial_inputs.shape, spatial_inputs.dtype)

for channel, name in enumerate(
    ["raw_norm", "foreground", "edt", "boundary", "marker_heatmap"]
):
    values = spatial_inputs[channel]
    print(
        f"{channel}: {name:16s} "
        f"min={float(values.min()):.5g} "
        f"max={float(values.max()):.5g} "
        f"finite={bool(np.isfinite(values).all())}"
    )


## 5. Build current-instance metadata and the full temporal graph

In [ ]:
instance_metadata = extract_instance_metadata(
    current_target,
    spatial_inputs[0],
    tuple(spacing),
    dref_um,
    spatial_inputs[4],
)

temporal = _build_temporal_inputs(
    track_graph,
    instance_movie,
    raw_movie,
    markers_movie,
    roi,
    roi_low,
    TARGET_LOCAL_T,
    spacing,
    dref_um,
    current_target,
)

target = build_gt_targets(
    gt_target,
    tuple(spacing),
    dref_um,
)

print("Current instances :", len(instance_metadata.ids))
print("Temporal nodes    :", len(temporal["graph_x"]))
print("Temporal tracklets:", len(temporal["temporal_ref_um"]))
print("GT instances      :", len(target["ids"]))

print(
    "Shape-feature max:",
    "instance=",
    float(instance_metadata.features[:, 7:9].max()),
    "graph=",
    float(temporal["graph_x"][:, 11:13].max()),
)

assert torch.isfinite(instance_metadata.features).all()
assert torch.isfinite(temporal["graph_x"]).all()


## 6. Assemble the single all-cell batch

In [ ]:
batch = {
    "spatial_inputs": torch.as_tensor(spatial_inputs).unsqueeze(0),
    "instance_labels": torch.as_tensor(current_target).unsqueeze(0),
    "spacing_um": torch.as_tensor(spacing).unsqueeze(0),
    "dref_um": torch.tensor([dref_um], dtype=torch.float32),
    "targets": [target],
    "instance_features": instance_metadata.features,
    "instance_ids": instance_metadata.ids,
    "instance_batch": torch.zeros(
        len(instance_metadata.ids),
        dtype=torch.long,
    ),
    "instance_centroids_um": instance_metadata.centroids_um,
    **temporal,
    "temporal_batch": torch.zeros(
        len(temporal["temporal_ref_um"]),
        dtype=torch.long,
    ),
}

cfg = _reduced_config()

required_queries = (
    2 * len(instance_metadata.ids)
    + len(temporal["temporal_ref_um"])
    + cfg.queries.discovery_queries
)

print("Required queries:", required_queries)
print("Query cap       :", cfg.queries.max_queries)
print("Target map      :", tuple(target["label_map"].shape), target["label_map"].dtype)
print("Dense GT masks  :", "masks" in target)


## 7. Instantiate the current source model

The reduced-width profile is the same profile used by the successful source-level acceptance test. No model modules are replaced manually here.


In [ ]:
model = StirNet(cfg).to(device)
criterion = RefinementCriterion(
    cfg.losses,
    cfg.queries,
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.training.lr,
    weight_decay=cfg.training.weight_decay,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=True,
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Trainable parameters:", f"{trainable_parameters:,}")
print("Learning rate        :", cfg.training.lr)
print("Weight decay         :", cfg.training.weight_decay)
print("Max grad norm        :", cfg.training.max_grad_norm)


## 8. Move model inputs to CUDA

Large target maps deliberately remain on CPU. The spatial volume is stored on CUDA as FP16, matching the successful acceptance run.


In [ ]:
batch_device = {}

for key, value in batch.items():
    if key == "targets":
        batch_device[key] = value
    elif key == "spatial_inputs":
        batch_device[key] = value.to(
            device=device,
            dtype=torch.float16,
        )
    elif key == "instance_labels":
        batch_device[key] = value.to(
            device=device,
            dtype=torch.int32,
        )
    else:
        batch_device[key] = move_to_device(
            value,
            device,
        )

print(
    "spatial_inputs:",
    batch_device["spatial_inputs"].shape,
    batch_device["spatial_inputs"].dtype,
    batch_device["spatial_inputs"].device,
)
print(
    "instance_labels:",
    batch_device["instance_labels"].dtype,
    batch_device["instance_labels"].device,
)
print(
    "target label_map:",
    batch_device["targets"][0]["label_map"].dtype,
    batch_device["targets"][0]["label_map"].device,
)


## 9. Clean forward + full loss gate

In [ ]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

model.eval()
criterion.eval()

forward_start = time.perf_counter()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
):
    outputs = model_forward_from_batch(
        model,
        batch_device,
    )
    forward_losses = criterion(
        outputs,
        batch_device["targets"],
    )

torch.cuda.synchronize()

forward_seconds = time.perf_counter() - forward_start
forward_peak_gib = torch.cuda.max_memory_allocated() / 1024**3

output_tensors = {
    "exist_logits": outputs.exist_logits,
    "centers_cellscale": outputs.centers_cellscale,
    "coarse_mask_logits": outputs.coarse_mask_logits,
}

for name, tensor in output_tensors.items():
    finite = bool(torch.isfinite(tensor.float()).all())
    print(
        f"{name:24s} "
        f"shape={tuple(tensor.shape)} "
        f"dtype={tensor.dtype} "
        f"finite={finite}"
    )
    if not finite:
        raise RuntimeError(f"{name} is non-finite.")

print("\nLosses")
print("------")
for name, value in forward_losses.items():
    finite = bool(torch.isfinite(value.float()).all())
    print(f"{name:18s}: {float(value):.7f}  finite={finite}")
    if not finite:
        raise RuntimeError(f"loss/{name} is non-finite.")

print(f"\nForward + loss time : {forward_seconds:.2f} s")
print(f"Forward peak CUDA    : {forward_peak_gib:.3f} GiB")


## 10. Release the evaluation graph before backward

The next cell performs a fresh training-mode forward so autograd can retain exactly the activations needed for backward.


In [ ]:
initial_eval_loss = float(forward_losses["loss"].detach().cpu())

del outputs
del forward_losses

gc.collect()
torch.cuda.empty_cache()

print("Initial eval loss:", initial_eval_loss)
print(
    "CUDA allocated after cleanup:",
    f"{torch.cuda.memory_allocated() / 1024**3:.3f} GiB",
)


## 11. One real AMP backward + optimizer step

This is the new gate.

It checks:

- finite training loss;
- successful `backward()`;
- finite gradients;
- non-zero gradients;
- finite global gradient norm;
- gradient clipping;
- optimizer parameter update;
- peak CUDA memory across forward + backward + step.

If this cell OOMs, the printed CUDA statistics identify the training-memory boundary for the next optimization pass.


In [ ]:
model.train()
criterion.train()
optimizer.zero_grad(set_to_none=True)

tracked_name, tracked_parameter = next(
    (name, parameter)
    for name, parameter in model.named_parameters()
    if parameter.requires_grad
)

tracked_before = (
    tracked_parameter
    .detach()
    .float()
    .cpu()
    .clone()
)

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

scale_before = float(scaler.get_scale())
step_start = time.perf_counter()

try:
    with torch.autocast(
        device_type="cuda",
        dtype=torch.float16,
    ):
        train_outputs = model_forward_from_batch(
            model,
            batch_device,
        )
        train_losses = criterion(
            train_outputs,
            batch_device["targets"],
        )
        train_loss = train_losses["loss"]

    if not torch.isfinite(train_loss.float()).all():
        raise RuntimeError(
            f"Training loss is non-finite before backward: {float(train_loss)}"
        )

    print("Training loss before backward:", float(train_loss.detach().cpu()))

    scaler.scale(train_loss).backward()

    # Convert scaled gradients back to their true magnitude before diagnostics
    # and clipping.
    scaler.unscale_(optimizer)

    grad_tensors = 0
    finite_grad_tensors = 0
    nonzero_grad_tensors = 0
    grad_sq_sum = 0.0
    grad_max_abs = 0.0

    for parameter in model.parameters():
        if parameter.grad is None:
            continue

        grad_tensors += 1
        grad = parameter.grad.detach().float()

        finite = bool(torch.isfinite(grad).all())
        finite_grad_tensors += int(finite)

        if not finite:
            raise RuntimeError(
                "A model gradient contains NaN or Inf."
            )

        max_abs = float(grad.abs().max())
        grad_max_abs = max(grad_max_abs, max_abs)

        if max_abs > 0:
            nonzero_grad_tensors += 1

        grad_sq_sum += float(
            grad.square().sum()
        )

    global_grad_norm = grad_sq_sum ** 0.5

    if grad_tensors == 0:
        raise RuntimeError("No parameter received a gradient.")

    if nonzero_grad_tensors == 0:
        raise RuntimeError("All parameter gradients are exactly zero.")

    if not np.isfinite(global_grad_norm):
        raise RuntimeError("Global gradient norm is non-finite.")

    clip_return = torch.nn.utils.clip_grad_norm_(
        model.parameters(),
        cfg.training.max_grad_norm,
    )

    if not torch.isfinite(
        torch.as_tensor(clip_return).float()
    ).all():
        raise RuntimeError(
            "clip_grad_norm_ returned a non-finite norm."
        )

    scaler.step(optimizer)
    scaler.update()

    torch.cuda.synchronize()

except torch.cuda.OutOfMemoryError:
    torch.cuda.synchronize()
    print("\nCUDA OOM during the backward gate")
    print(
        "allocated:",
        f"{torch.cuda.memory_allocated() / 1024**3:.3f} GiB",
    )
    print(
        "reserved :",
        f"{torch.cuda.memory_reserved() / 1024**3:.3f} GiB",
    )
    print(
        "peak     :",
        f"{torch.cuda.max_memory_allocated() / 1024**3:.3f} GiB",
    )
    raise

step_seconds = time.perf_counter() - step_start
step_peak_gib = torch.cuda.max_memory_allocated() / 1024**3
scale_after = float(scaler.get_scale())

tracked_after = (
    tracked_parameter
    .detach()
    .float()
    .cpu()
)

tracked_delta = float(
    (tracked_after - tracked_before)
    .abs()
    .max()
)

print("\nBackward gate")
print("-------------")
print("Tracked parameter       :", tracked_name)
print("Tracked max |delta|      :", tracked_delta)
print("Gradient tensors        :", grad_tensors)
print("Finite gradient tensors :", finite_grad_tensors)
print("Nonzero gradient tensors:", nonzero_grad_tensors)
print("Global grad L2 norm     :", global_grad_norm)
print("Max |gradient|          :", grad_max_abs)
print("clip_grad_norm_ return  :", float(clip_return))
print("GradScaler before/after :", scale_before, "->", scale_after)
print("Forward+backward+step   :", f"{step_seconds:.2f} s")
print("Peak CUDA allocation    :", f"{step_peak_gib:.3f} GiB")

if tracked_delta == 0.0:
    raise RuntimeError(
        "The tracked parameter did not change after optimizer.step()."
    )

print("\nONE BACKWARD + OPTIMIZER STEP PASSED.")


## 12. Release gradients and training graph

This frees training activations before the optional post-step forward sanity check.


In [ ]:
optimizer.zero_grad(set_to_none=True)

del train_outputs
del train_losses
del train_loss

gc.collect()
torch.cuda.empty_cache()

print(
    "CUDA allocated after backward cleanup:",
    f"{torch.cuda.memory_allocated() / 1024**3:.3f} GiB",
)


## 13. Post-step finite sanity check

This is still only a single optimizer step. It verifies that the updated model remains numerically valid.


In [ ]:
model.eval()
criterion.eval()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
):
    post_outputs = model_forward_from_batch(
        model,
        batch_device,
    )
    post_losses = criterion(
        post_outputs,
        batch_device["targets"],
    )

post_loss = float(post_losses["loss"].detach().cpu())

for name, tensor in {
    "exist_logits": post_outputs.exist_logits,
    "centers_cellscale": post_outputs.centers_cellscale,
    "coarse_mask_logits": post_outputs.coarse_mask_logits,
}.items():
    if not torch.isfinite(tensor.float()).all():
        raise RuntimeError(
            f"Post-step {name} is non-finite."
        )

for name, value in post_losses.items():
    if not torch.isfinite(value.float()).all():
        raise RuntimeError(
            f"Post-step loss/{name} is non-finite."
        )

print("Initial eval loss :", initial_eval_loss)
print("Post-step eval loss:", post_loss)
print("All post-step outputs and losses are finite.")

# A single optimizer step is not expected to guarantee a lower eval loss.
# The next experiment, after this gate passes, will be repeated same-sample overfitting.


## Stop here

If the backward cell passes, the next notebook/task should be the repeated same-sample overfit experiment.

Do **not** infer generalization from this one sample. The purpose of the next stage will only be to prove that the model can optimize the training objective on a fixed all-cell scene.
